# L30 Lab: CFAR Processing, notebook

This notebook runs the course's CFAR detector, `cfar_2d` in `radar_lab.py`, on range-Doppler maps like the ones you made in the L29 Lab. You do not write the detector; you run it, change what each task names, and record what you see in Tables D to F of the L30 Lab packet. There are three tasks, (a) to (c), about 30 to 45 minutes in all, and seven changed lines between them.

**To run it:** use the folder you set up for the L28 Lab (the L28 page's Notebook Setup frame): this notebook, `radar_lab.py`, and the captures in one folder, or the course's layout (`notebooks/` beside `radar-data/`). Run the cells in order from the top.

The course's recorded data (`L30_*.npz`) works in every cell. Each file is one burst of the L29 waveform: 256 chirps of 300 µs, 0.5 ms apart, 500 MHz, so the map's cells are 0.30 m by 0.12 m/s. To run a task on your own L29 or L30 burst instead, change the file name in the next cell.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage, stats
import radar_lab as rl

# Where the captures are: the course's recorded data if this folder sits beside
# radar-data/, otherwise this folder.
DATA = Path("../radar-data") if Path("../radar-data").exists() else Path(".")

empty = rl.load_capture(DATA / "L30_empty.npz")              # task (a): no target
wall_hand = rl.load_capture(DATA / "L30_wall_hand.npz")      # tasks (b) and (c)
weak_strong = rl.load_capture(DATA / "L30_weak_strong.npz")  # task (b)

for name, cap in (("empty", empty), ("wall_hand", wall_hand), ("weak_strong", weak_strong)):
    print(f"{name:12s} {cap['iq'].shape[0]} chirps of {cap['iq'].shape[1]} samples, "
          f"B = {cap['B']/1e6:.0f} MHz, PRI = {cap['PRI']*1e3:.2f} ms, f0 = {cap['f0']/1e3:.1f} kHz")

In [ ]:
# Helpers used by the tasks. Run this cell; you do not need to change it.
def n_ref(guard, ref):
    """Reference cells in cfar_2d's ring for (range, Doppler) half-widths."""
    (gr, gv), (rr, rv) = guard, ref
    return (2 * (gr + rr) + 1) * (2 * (gv + rv) + 1) - (2 * gr + 1) * (2 * gv + 1)

def quiet_cells(rng, vel, guard, ref):
    """Cells that can only hold noise in an empty scene: beyond 1 m, and far
    enough from 0 m/s that no reference cell reaches the kit's own leakage."""
    j0 = np.argmin(np.abs(vel))
    far_from_zero = np.abs(np.arange(vel.size) - j0) > guard[1] + ref[1] + 1
    return (rng > 1.0)[:, None] & far_from_zero[None, :]

def targets(det, hits):
    """One entry per cluster of touching detected cells: its strongest cell and
    the cluster's size, as (range, velocity, snr_db, n_cells), strongest first."""
    lab, _ = ndimage.label(det)
    best = {}
    for (i, j), h in zip(zip(*np.nonzero(det)), hits):     # hits run in this order
        k = lab[i, j]
        if k not in best or h[2] > best[k][2]:
            best[k] = (*h, int((lab == k).sum()))
    return sorted(best.values(), key=lambda t: -t[2])

def show(ax, rd, rng, vel, det, title, r_max=8.0):
    """The map in dBFS, with each detected cell marked."""
    keep = rng <= r_max
    ax.pcolormesh(vel, rng[keep], rd[keep], vmin=-120, vmax=-30, cmap="Blues", shading="auto")
    i, j = np.nonzero(det & keep[:, None])
    ax.plot(vel[j], rng[i], "s", mfc="none", mec="red", ms=4)
    ax.set_xlim(-4, 4); ax.set_xlabel("velocity (m/s), + closing"); ax.set_ylabel("range (m)")
    ax.set_title(title, fontsize=10)

## Task (a): set the bias for a false-alarm rate, then count

`L30_empty.npz` holds no target, only noise and the kit's own leakage at 0 m. Every detection in it is a false alarm, apart from the leakage itself, so the cell counts detections only where no reference cell can reach the leakage: beyond 1 m and away from 0 m/s. Those are 136 406 cells.

1. **Change one line:** replace `None` in `BIAS_DB = None` with the bias, in dB, that gives `PFA` with `N` reference cells (the bias equation of the lesson's "Bias From the False-Alarm Probability" frame; your Pre-Lab Question 1). Run the cell with `PFA = 1e-3` and record the design count, the measured count, and the measured rate in Table D.
2. **Change one line:** set `PFA = 1e-4`, run it, and record the second row.
3. **Change one line:** set `WINDOW = None`, the rectangular window, with `PFA` back at `1e-3`, and record the third row.

**Check:** `BIAS_DB` prints as 8.50 dB at $10^{-3}$ and 9.78 dB at $10^{-4}$. With the Hann window the measured rate comes out above the design: about 1.3 times it at $10^{-3}$ and about twice it at $10^{-4}$. With the rectangular window at $10^{-3}$ it lands within about 10% of the design. Part 3 of the packet asks you why the window changes the count when nothing else did. A count of a dozen or so has a scatter of about its own square root, so read the $10^{-4}$ row with that in mind.

In [ ]:
GUARD = (2, 2)           # guard cells each side, (range, Doppler)
REF = (4, 4)             # reference cells each side beyond the guard cells, (range, Doppler)
PFA = 1e-3               # TASK (a), step 2: change to 1e-4
WINDOW = "hann"          # TASK (a), step 3: change to None, with PFA back at 1e-3
N = n_ref(GUARD, REF)
BIAS_DB = None           # TASK (a), step 1: the bias for PFA with N reference cells, in dB

rd, rng, vel = rl.range_doppler(empty, window=WINDOW)
test = quiet_cells(rng, vel, GUARD, REF)
if BIAS_DB is None:
    print(f"N = {N} reference cells. Set BIAS_DB (step 1) and run this cell again.")
else:
    det, _ = rl.cfar_2d(rd, GUARD, REF, BIAS_DB)
    n_fa, n_test = int(det[test].sum()), int(test.sum())
    print(f"window {WINDOW}, N = {N}, BIAS_DB = {BIAS_DB:.2f} dB, design PFA {PFA:.0e}")
    print(f"  {n_test} cells tested: design {PFA * n_test:.0f} false alarms, measured {n_fa}, "
          f"rate {n_fa / n_test:.2e} ({n_fa / n_test / PFA:.2f} x design)")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    show(ax, rd, rng, vel, det & test, f"L30_empty: false alarms at design PFA {PFA:.0e}", r_max=40)
    ax.set_xlim(vel[0], vel[-1]); plt.show()

## Task (b): cell averaging, greatest-of, smallest-of

Two scenes, the same detector settings: 2 guard and 4 reference cells each way and an 11.6 dB bias, the lesson's 2D example. The bias is held fixed so that only the method changes; strictly, each method needs its own bias for the same $P_{fa}$ (the lesson's "Greatest-Of and Smallest-Of" frame).

- `L30_wall_hand.npz`: a wall at 5.0 m, square to the beam, and a hand waving at 2.4 m. The wall stands still, so it sits at 0 m/s with the leakage.
- `L30_weak_strong.npz`: a strong mover at 4.2 m and a weak one at 3.0 m, four range cells nearer, at almost the same velocity: the lesson's masking case.

The cell prints one line per cluster of detected cells: its strongest cell's range, velocity, and SNR over its own noise estimate, and how many cells the cluster holds.

1. Run it as it stands (`METHOD = "ca"`). For each scene, record in Table E the clusters at the wall (or the strong target), at the hand (or the weak target), and anywhere else.
2. **Change one line:** set `METHOD = "go"` and run it again. Then set `METHOD = "so"` and run it a third time.

**Check:** with `"ca"` the wall is one cluster of nine cells, 3 range by 3 velocity cells, the hand is found, one false alarm sits far out near 78 m, and the weak target is missing. `"go"` gives the same wall and hand clusters, loses the far false alarm, and still misses the weak target. `"so"` finds the weak target, but the wall's cluster doubles to 19 cells, the hand's grows from 6 cells to 22, the strong target's from 7 to 21, and five clusters of false alarms appear, four of them new. Look at where the extra cells sit in the plot; Part 3 asks you to explain each change.

In [ ]:
METHOD = "ca"            # TASK (b): change to "go", then to "so"

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, cap) in zip(axs, (("L30_wall_hand", wall_hand), ("L30_weak_strong", weak_strong))):
    rd, rng, vel = rl.range_doppler(cap)
    det, hits = rl.cfar_2d(rd, (2, 2), (4, 4), 11.6, axes=(rng, vel), method=METHOD)
    print(f"{name}, method {METHOD}: {int(det.sum())} cells detected, in these clusters:")
    for r, v, snr, n in sorted(targets(det, hits)):
        print(f"   {r:6.2f} m  {v:+6.2f} m/s  {snr:5.1f} dB  {n:3d} cells")
    show(ax, rd, rng, vel, det, f"{name}, method {METHOD}")
plt.tight_layout(); plt.show()

## Task (c): the probability of detection, and the list for L31

The hand in `L30_wall_hand.npz` stands far above the noise, so CFAR finds it at any bias you would use: one map says $P_d = 1$ and nothing else. $P_d$ only moves at an SNR near the 13 dB of Lesson 27, and measuring it takes many maps of the same target. The cell below makes them: it adds fresh noise to the capture `TRIALS` times, enough to bring the hand's strongest cell down to `SNR_DB` above the noise, as if the hand stood much farther away, and counts the maps in which `cfar_2d` still declares that cell. It prints the count beside two predictions: Lesson 27's $P_d$ for a steady target with the noise power known, and the same with the noise estimated from $N$ independent reference cells, which includes the CFAR loss.

1. Run it as it stands, at `BIAS_DB_C = 11.6`, the bias for $10^{-6}$. Record the measured $P_d$ and the two predictions in Table F.
2. **Change one line:** set `BIAS_DB_C` to your task (a) bias for $10^{-3}$ and run it again. Record the second row.

**Check:** at 12 dB the measured $P_d$ is near 0.6 at the $10^{-6}$ bias and above 0.9 at the $10^{-3}$ bias, each within about 0.1 of the CFAR prediction. With 50 maps, a measured $P_d$ has a scatter of about 0.07. The cell takes about half a minute.

In [ ]:
BIAS_DB_C = 11.6         # TASK (c), step 2: change to your task (a) bias for 1e-3
SNR_DB = 12.0            # the hand's strongest cell over the noise, after the added noise
TRIALS = 50

cap = wall_hand
rd0, rng, vel = rl.range_doppler(cap)
hand = ((rng > 1.5) & (rng < 3.5))[:, None] & (np.abs(vel) > 0.3)[None, :]
ih, jh = np.unravel_index(np.argmax(np.where(hand, rd0, -300)), rd0.shape)

# Noise per sample that puts the hand's cell SNR_DB above the noise in its cell.
# The two Hann FFTs pass 1.5/n of the noise power each (L28, the window's noise bandwidth).
m, n = cap["iq"].shape
p_cell = 10 ** ((rd0[ih, jh] - SNR_DB) / 10)
sigma = np.sqrt(p_cell / (1.5 / n * 1.5 / m) / 2)           # per real and imaginary part
rng_noise = np.random.default_rng(30)
found = 0
for _ in range(TRIALS):
    noisy = {**cap, "iq": cap["iq"] + sigma * (rng_noise.standard_normal((m, n))
                                               + 1j * rng_noise.standard_normal((m, n)))}
    rd, _, _ = rl.range_doppler(noisy)
    det, _ = rl.cfar_2d(rd, (2, 2), (4, 4), BIAS_DB_C)
    found += bool(det[ih, jh])

# Lesson 27's steady-target Pd (Marcum Q), with the noise known and with CA-CFAR's
# estimate from N independent cells: the threshold alpha * x, x the estimate over Pn.
alpha, snr, N = 10 ** (BIAS_DB_C / 10), 10 ** (SNR_DB / 10), n_ref((2, 2), (4, 4))
pd_known = stats.ncx2.sf(2 * alpha, 2, 2 * snr)
x = np.linspace(0.3, 2.5, 2001)
pd_cfar = np.sum(stats.ncx2.sf(2 * alpha * x, 2, 2 * snr) * stats.gamma.pdf(x, N, scale=1 / N)) * (x[1] - x[0])
print(f"hand cell {rng[ih]:.2f} m, {vel[jh]:+.2f} m/s, {rd0[ih, jh]:.1f} dBFS; set to {SNR_DB:.0f} dB SNR")
print(f"BIAS_DB_C = {BIAS_DB_C:.2f} dB: found in {found} of {TRIALS} maps, Pd = {found / TRIALS:.2f}")
print(f"  Lesson 27, noise known: Pd = {pd_known:.2f};  CA-CFAR, N = {N}: Pd = {pd_cfar:.2f}")

### The detection list you keep for L31

The last cell makes the list the L31 Lab classifies: one entry per object, from the wall-and-hand capture or your own burst, after the 2-pulse canceller of Lesson 29 has removed what stands still. As shipped it prints the list in cell indices.

**Change one line:** add `axes=(rng, vel)` to the `cfar_2d` call, so the list comes back in meters and meters per second, and run it. It writes `L30_detections.csv` beside this notebook.

**Check:** two entries. The first is the hand, near 2.6 m and $-0.8$ m/s: opening, because the burst catches the hand moving away from the kit. The second is a single cell near 11 m with an SNR of 11.9 dB, just over the 11.6 dB bias: a false alarm. The wall and the leakage are gone with the canceller. Record both entries in Table F, mark which is the target, and keep the file with your captures.

In [ ]:
LIST_CAPTURE = wall_hand        # or rl.load_capture("your_burst.npz")

rd, rng, vel = rl.range_doppler(rl.mti(LIST_CAPTURE, "2pulse"))
det, hits = rl.cfar_2d(rd, (2, 2), (4, 4), 11.6)   # TASK (c), step 3: add axes=(rng, vel)
found_list = targets(det, hits)
for r, v, snr, n in found_list:
    print(f"range {r:7.2f}   velocity {v:+7.2f}   SNR {snr:5.1f} dB   {n} cells")
np.savetxt("L30_detections.csv", [t[:3] for t in found_list], delimiter=",", fmt="%.3f",
           header="range_m,vel_mps,snr_db", comments="")

## Before you close

Save the notebook with your changes and your plots, and turn it in with the packet. Keep it, `L30_detections.csv`, `radar_lab.py`, and your captures together: the L31 Lab starts from the detection list. Anything you do beyond these three tasks, such as another guard band, a scene of your own, or the bias each method needs for the same $P_{fa}$, belongs in your report's Extra Effort.